## 1. Setup and Imports

In [ ]:
import os
from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from datetime import datetime

from oceanstream.geotrack.processor import convert
from oceanstream.providers import get_provider
from oceanstream.sensors.processors.nmea_gnss import process_nmea_raw

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✓ Imports successful")

## 2. Inspect Raw NMEA Data

Let's examine a sample NMEA file to understand its structure.

In [ ]:
# Path to NMEA file (adjust to your data location)
nmea_file = Path("raw_data/gnss_log.txt")

# Check if file exists
if not nmea_file.exists():
    print(f"⚠️  File not found: {nmea_file}")
    print(f"Please adjust the path to point to your NMEA data file")
else:
    print(f"✓ Found NMEA file: {nmea_file}")
    file_size_mb = nmea_file.stat().st_size / 1024 / 1024
    print(f"  Size: {file_size_mb:.2f} MB")

In [ ]:
# Display first 20 lines
print("First 20 lines of NMEA file:\n")
print("="*80)

with open(nmea_file, 'r') as f:
    for i, line in enumerate(f, 1):
        if i > 20:
            break
        print(f"{i:3d}: {line.rstrip()}")

print("="*80)

In [ ]:
# Analyze NMEA sentence types
sentence_counts = {}
total_lines = 0
has_timestamps = False

with open(nmea_file, 'r') as f:
    for line in f:
        total_lines += 1
        
        # Check for ISO 8601 timestamp prefix
        if line[0:4].isdigit() and 'T' in line[:20]:
            has_timestamps = True
        
        # Extract sentence type (e.g., GGA, RMC)
        if '$' in line:
            parts = line.split('$', 1)[1].split(',', 1)
            if parts:
                sentence_type = parts[0][2:]  # Remove GP/GN prefix
                sentence_counts[sentence_type] = sentence_counts.get(sentence_type, 0) + 1

print(f"\nNMEA File Statistics:\n")
print(f"  Total lines: {total_lines:,}")
print(f"  Has external timestamps: {'Yes' if has_timestamps else 'No'}")
print(f"\n  Sentence Types:")
print(f"  {'Type':<10} {'Count':>10} {'Percentage':>12}")
print(f"  {'-'*10} {'-'*10} {'-'*12}")

for sentence_type, count in sorted(sentence_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_lines) * 100
    print(f"  {sentence_type:<10} {count:>10,} {percentage:>11.1f}%")

print(f"  {'-'*10} {'-'*10} {'-'*12}")
print(f"  {'TOTAL':<10} {total_lines:>10,} {100.0:>11.1f}%")

## 3. Basic NMEA Processing

Process the NMEA file into GeoParquet format.

In [ ]:
# Configuration
input_source = Path("raw_data/gnss_log.txt")
output_dir = Path("out/geoparquet")
campaign_id = "voyage_2024"

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)

# Get provider
provider = get_provider("generic")

print(f"Processing NMEA file: {input_source}")
print(f"Output directory: {output_dir}")
print(f"Campaign ID: {campaign_id}")
print("\nStarting processing...\n")

# Process NMEA file
convert(
    provider=provider,
    input_source=input_source,
    output_dir=output_dir,
    campaign_id=campaign_id,
    verbose=True,
    yes=True  # Skip confirmation prompt
)

print(f"\n✓ Processing complete!")
print(f"Output: {output_dir / campaign_id}")

## 4. Load and Explore GeoParquet Data

In [ ]:
# Load GeoParquet dataset
campaign_dir = output_dir / campaign_id
gdf = gpd.read_parquet(campaign_dir)

print(f"Loaded GeoParquet dataset: {len(gdf):,} rows\n")
print("Schema:")
print(gdf.dtypes)
print(f"\nMemory usage: {gdf.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# Display dataset summary
print("Dataset Summary:\n")
print(f"  Total rows: {len(gdf):,}")
print(f"  Columns: {len(gdf.columns)}")
print(f"\n  Temporal Extent:")
print(f"    Start: {gdf['time'].min()}")
print(f"    End:   {gdf['time'].max()}")
duration = (gdf['time'].max() - gdf['time'].min()).total_seconds()
print(f"    Duration: {duration/3600:.2f} hours ({duration/86400:.2f} days)")

print(f"\n  Spatial Extent:")
print(f"    Latitude:  {gdf['latitude'].min():>8.4f}° to {gdf['latitude'].max():>8.4f}°")
print(f"    Longitude: {gdf['longitude'].min():>8.4f}° to {gdf['longitude'].max():>8.4f}°")

if 'altitude' in gdf.columns:
    print(f"    Altitude:  {gdf['altitude'].min():>8.1f}m to {gdf['altitude'].max():>8.1f}m")

print(f"\n  Available Columns:")
for col in gdf.columns:
    print(f"    - {col}")

In [ ]:
# Display first few rows
print("First 5 rows:\n")
gdf.head()

In [ ]:
# Statistical summary
print("Statistical Summary:\n")
gdf.describe()

## 5. Visualize Track

In [ ]:
# Plot vessel track
fig, ax = plt.subplots(figsize=(14, 10))

# Plot track
gdf.plot(ax=ax, marker='o', markersize=2, color='blue', alpha=0.6, linewidth=0.5)

# Add start and end markers
start_point = gdf.iloc[0]
end_point = gdf.iloc[-1]
ax.plot(start_point.geometry.x, start_point.geometry.y, 'go', markersize=10, label='Start', zorder=5)
ax.plot(end_point.geometry.x, end_point.geometry.y, 'ro', markersize=10, label='End', zorder=5)

# Labels and styling
ax.set_xlabel('Longitude (°)', fontsize=12)
ax.set_ylabel('Latitude (°)', fontsize=12)
ax.set_title('GNSS Track from NMEA Data', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='best', fontsize=10)

# Add info box
info_text = f"Total points: {len(gdf):,}\n"
info_text += f"Duration: {duration/3600:.1f} hours\n"
info_text += f"Start: {gdf['time'].min().strftime('%Y-%m-%d %H:%M')}"
ax.text(0.02, 0.98, info_text, transform=ax.transAxes, 
        fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Plot speed over time (if speed data available)
if 'speed_knots' in gdf.columns:
    fig, ax = plt.subplots(figsize=(14, 5))
    
    ax.plot(gdf['time'], gdf['speed_knots'], linewidth=0.8, color='navy', alpha=0.7)
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Speed (knots)', fontsize=12)
    ax.set_title('Vessel Speed Over Time', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Add statistics
    mean_speed = gdf['speed_knots'].mean()
    max_speed = gdf['speed_knots'].max()
    ax.axhline(y=mean_speed, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'Mean: {mean_speed:.2f} kts')
    ax.legend(loc='best')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Speed Statistics:")
    print(f"  Mean: {mean_speed:.2f} knots")
    print(f"  Max:  {max_speed:.2f} knots")
    print(f"  Min:  {gdf['speed_knots'].min():.2f} knots")

## 6. Advanced: NMEA Processing with Filtering

Process NMEA data with sentence type filtering and decimation.

In [ ]:
# Configuration for filtered processing
input_source = Path("raw_data/gnss_log.txt")
output_dir = Path("out/geoparquet")
campaign_id = "voyage_2024_filtered"

# Get provider
provider = get_provider("generic")

print("Processing NMEA with filtering:")
print("  Sentence types: GGA, RMC only")
print("  Sampling interval: 10 seconds")
print("\nStarting processing...\n")

# Process with filtering
convert(
    provider=provider,
    input_source=input_source,
    output_dir=output_dir,
    campaign_id=campaign_id,
    nmea_sentence_types=["GGA", "RMC"],  # Only GGA and RMC
    nmea_sampling_interval=10.0,          # 1 point per 10 seconds
    verbose=True,
    yes=True
)

print(f"\n✓ Filtered processing complete!")

In [ ]:
# Compare original vs. filtered
gdf_original = gpd.read_parquet(output_dir / "voyage_2024")
gdf_filtered = gpd.read_parquet(output_dir / "voyage_2024_filtered")

print("\nComparison: Original vs. Filtered\n")
print(f"  Original dataset:")
print(f"    Rows: {len(gdf_original):,}")
print(f"    Memory: {gdf_original.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

print(f"\n  Filtered dataset (GGA+RMC, 10s sampling):")
print(f"    Rows: {len(gdf_filtered):,}")
print(f"    Memory: {gdf_filtered.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

reduction = (1 - len(gdf_filtered) / len(gdf_original)) * 100
print(f"\n  Reduction: {reduction:.1f}%")
print(f"  Speedup: {len(gdf_original) / len(gdf_filtered):.1f}x fewer points")

## 7. Export for Further Analysis

In [ ]:
# Export to CSV for external tools
output_csv = Path("out/temp/track_export.csv")
output_csv.parent.mkdir(parents=True, exist_ok=True)

# Select columns to export
export_columns = ['time', 'latitude', 'longitude', 'altitude', 'speed_knots', 'course']
export_columns = [col for col in export_columns if col in gdf.columns]

gdf[export_columns].to_csv(output_csv, index=False)
print(f"✓ Exported to CSV: {output_csv}")
print(f"  Rows: {len(gdf):,}")
print(f"  Columns: {', '.join(export_columns)}")

In [ ]:
# Export to GeoJSON for web mapping
output_geojson = Path("out/temp/track.geojson")

# Sample data if too large (keep every 10th point)
if len(gdf) > 10000:
    gdf_export = gdf.iloc[::10]
    print(f"⚠️  Large dataset - sampling to {len(gdf_export):,} points (every 10th)")
else:
    gdf_export = gdf

gdf_export.to_file(output_geojson, driver='GeoJSON')
print(f"✓ Exported to GeoJSON: {output_geojson}")
print(f"  Size: {output_geojson.stat().st_size / 1024 / 1024:.2f} MB")

## Summary

This notebook demonstrated:

1. ✓ Inspecting raw NMEA data structure
2. ✓ Processing NMEA files into GeoParquet
3. ✓ Analyzing spatial and temporal characteristics
4. ✓ Visualizing vessel tracks
5. ✓ Advanced filtering and decimation
6. ✓ Exporting for further analysis

## Next Steps

- **Explore STAC metadata**: Check `out/geoparquet/{campaign_id}/stac/collection.json`
- **Query with DuckDB**: Use SQL for fast spatial queries
- **Generate PMTiles**: Add `--generate-pmtiles` for web visualization
- **Process multiple files**: Point to directory with multiple NMEA logs

## Resources

- [NMEA Processing Guide](../core-concepts/nmea-processing.md)
- [CLI Reference](../core-concepts/geotrack-convert-reference.md)
- [OceanStream Documentation](https://oceanstream.io/docs)